In [1]:
!pip install -q transformers datasets evaluate jiwer huggingface_hub inflect

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 52.3 MB/s eta 0:00:00a 0:00:01


In [2]:
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/test/cu121

Looking in indexes: https://download.pytorch.org/whl/test/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.0/781.0 MB 2.0 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 2.5 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 40.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 80.1 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 42.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 95.2 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.1 MB/s eta 0:00:000:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.0 MB/s eta 0:00:000:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 14.3 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 30.9 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━

In [3]:
import torch, torchaudio
print(torch.__version__)
print(torchaudio.__version__)

2.3.1+cu121
2.3.1+cu121


In [4]:
import os
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
import torchaudio
import pandas as pd

In [5]:
# Define the path to the main directory containing the subfolders
main_directory = "/kaggle/input/pakistani-english-dataset/WAV"
# List all subfolders inside the main directory
subfolders = [f.path for f in os.scandir(main_directory) if f.is_dir()]

# Initialize a list to store the full paths of .wav files
wav_files_traning = []
wav_files_testing = []

# Loop over each subfolder
for subfolder in subfolders:
    # Get the list of all files in the current subfolder
    # Filter the files that end with '.wav' and join the subfolder path to the filename
    wav_files_in_subfolder = [os.path.join(subfolder, file) for file in os.listdir(subfolder) if file.endswith('.wav')]

    X_train, X_test = train_test_split(wav_files_in_subfolder, test_size=0.1, random_state=42)

    wav_files_traning.extend(X_train)
    wav_files_testing.extend(X_test)

In [6]:
print("Length of traning data : ", len(wav_files_traning))
print("Length of testing data : ", len(wav_files_testing))

Length of traning data :  1969
Length of testing data :  222


In [7]:
transcription_data = pd.read_csv("/kaggle/input/pakistani-english-dataset/UTTERANCEINFO.txt", delimiter='\t')
transcription_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2191 entries, 0 to 2190
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   CHANNEL        2191 non-null   object
 1   UTTRANS_ID     2191 non-null   object
 2   SPEAKER_ID     2191 non-null   object
 3   PROMPT         2191 non-null   object
 4   TRANSCRIPTION  2191 non-null   object
dtypes: object(5)
memory usage: 85.7+ KB


In [8]:
transcription_data = transcription_data[["UTTRANS_ID", "TRANSCRIPTION"]]

In [9]:
def load_data(wav_files):
  audio_data = []

  for wav_path in wav_files:

    wav_id = os.path.basename(wav_path)
    transcription = transcription_data[transcription_data['UTTRANS_ID'] == wav_id]['TRANSCRIPTION'].values[0]

    # Check if the audio file exists
    if os.path.exists(wav_path):
        # Load audio using torchaudio
        waveform, sample_rate = torchaudio.load(wav_path)
        audio_np = waveform.squeeze().numpy()  # Convert the tensor to a NumPy array


        audio_data.append({
            'id': wav_id,
            'audio': audio_np,
            'sample_rate': sample_rate,
            'transcription': transcription,
        })
  return audio_data

In [10]:
traning_data = load_data(wav_files_traning)
testing_data = load_data(wav_files_testing)

traning_data = Dataset.from_pandas(pd.DataFrame(traning_data))
testing_data = Dataset.from_pandas(pd.DataFrame(testing_data))

dataset_dict = DatasetDict({
    'train': traning_data,
    'test': testing_data
})

In [11]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['id', 'audio', 'sample_rate', 'transcription'],
        num_rows: 1969
    })
    test: Dataset({
        features: ['id', 'audio', 'sample_rate', 'transcription'],
        num_rows: 222
    })
})

In [12]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small.en"
)

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.41M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

In [13]:
def prepare_dataset(example):
    audio = example["audio"]
    sampling_rate = example["sample_rate"]

    # compute log-Mel input features from input audio array
    example = processor(
        audio=audio,
        sampling_rate=example["sample_rate"],
        text=example["transcription"],
    )

    # compute input length of audio sample in seconds
    example["input_length"] = len(audio) / sampling_rate

    return example

In [14]:
dataset = dataset_dict.map(
    prepare_dataset, remove_columns=dataset_dict.column_names["train"], num_proc=1
)

Map:   0%|          | 0/1969 [00:00<?, ? examples/s]

Map:   0%|          | 0/222 [00:00<?, ? examples/s]

In [15]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels', 'input_length'],
        num_rows: 1969
    })
    test: Dataset({
        features: ['input_features', 'labels', 'input_length'],
        num_rows: 222
    })
})

In [16]:
max_input_length = 30.0


def is_audio_in_length_range(length):
    return length < max_input_length

In [17]:
dataset["train"] = dataset["train"].filter(
    is_audio_in_length_range,
    input_columns=["input_length"],
)

Filter:   0%|          | 0/1969 [00:00<?, ? examples/s]

In [18]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(
        self, features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [
            {"input_features": feature["input_features"][0]} for feature in features
        ]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

In [19]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [20]:
import evaluate

metric = evaluate.load("wer")

In [21]:
import inflect
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

normalizer = BasicTextNormalizer()
engine = inflect.engine()

# Custom function to convert digits to words
def numbers_to_words(text):
    words = []
    for word in text.split():
        if word.isdigit():
            words.append(engine.number_to_words(word))
        else:
            words.append(word)
    return " ".join(words)

# Updated compute_metrics function
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 with the pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # Decode predictions and references
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    # Orthographic WER (no normalization)
    wer_ortho = 100 * metric.compute(predictions=pred_str, references=label_str)

    # Apply basic normalization
    pred_str_norm = [normalizer(pred) for pred in pred_str]
    label_str_norm = [normalizer(label) for label in label_str]

    # Convert numbers to words
    pred_str_norm = [numbers_to_words(pred) for pred in pred_str_norm]
    label_str_norm = [numbers_to_words(label) for label in label_str_norm]

    # Filter non-zero references
    pred_str_norm = [
        pred_str_norm[i] for i in range(len(pred_str_norm)) if len(label_str_norm[i]) > 0
    ]
    label_str_norm = [
        label_str_norm[i]
        for i in range(len(label_str_norm))
        if len(label_str_norm[i]) > 0
    ]

    # Normalized WER
    wer = 100 * metric.compute(predictions=pred_str_norm, references=label_str_norm)

    return {"wer_ortho": wer_ortho, "wer": wer}


In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperConfig

# Load the model with the updated configuration
model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-base.en",
)

In [61]:
from functools import partial

# disable cache during training since it's incompatible with gradient checkpointing
model.config.use_cache = False

# set language and task for generation and re-enable cache
model.generate = partial(
    model.generate, use_cache=True)

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./finetune-whisper-base.en", 
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    lr_scheduler_type="linear",
    weight_decay=0.05,
    #max_grad_norm=1.0,
    warmup_steps=50,
    max_steps=500, 
    gradient_checkpointing=True,
    fp16=True,
    fp16_full_eval=True,
    eval_strategy="steps",
    per_device_eval_batch_size=16,
    predict_with_generate=True,
    #generation_max_length=225,
    save_steps=100,
    eval_steps=100,
    logging_steps=100,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)


In [64]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor,
)

max_steps is given, it will override any value given in num_train_epochs


In [65]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss,Validation Loss,Wer Ortho,Wer
100,0.780100,0.381192,13.355454,9.419407
200,0.281700,0.364731,12.567399,9.135201
300,0.185300,0.370513,12.774782,8.850995
400,0.146000,0.373778,12.567399,9.094600
500,0.109700,0.379690,12.816259,9.135201


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 357, 366, 438, 532, 685, 705, 796, 930, 1058, 1220, 1267, 1279, 1303, 1343, 1377, 1391, 1635, 1782, 1875, 2162, 2361, 2488, 3467, 4008, 4211, 4600, 4808, 5299, 5855, 6329, 7203, 9609, 9959, 10563, 10786, 11420, 11709, 11907, 13163, 13697, 13700, 14808, 15306, 16410, 16791, 17992, 19203, 19510, 20724, 22305, 22935, 27007, 30109, 30420, 33409, 34949, 40283, 40493, 40549, 47282, 49146, 50257, 50357, 50358, 50359, 50360, 50361], 'begin_suppress_tokens': [220, 50256]}
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:464: UserWarnin

TrainOutput(global_step=500, training_loss=0.300567569732666, metrics={'train_runtime': 1756.0578, 'train_samples_per_second': 4.556, 'train_steps_per_second': 0.285, 'total_flos': 5.149883695104e+17, 'train_loss': 0.300567569732666, 'epoch': 4.032258064516129})

In [68]:
from huggingface_hub import notebook_login

notebook_login()

In [69]:
trainer.push_to_hub()

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 357, 366, 438, 532, 685, 705, 796, 930, 1058, 1220, 1267, 1279, 1303, 1343, 1377, 1391, 1635, 1782, 1875, 2162, 2361, 2488, 3467, 4008, 4211, 4600, 4808, 5299, 5855, 6329, 7203, 9609, 9959, 10563, 10786, 11420, 11709, 11907, 13163, 13697, 13700, 14808, 15306, 16410, 16791, 17992, 19203, 19510, 20724, 22305, 22935, 27007, 30109, 30420, 33409, 34949, 40283, 40493, 40549, 47282, 49146, 50257, 50357, 50358, 50359, 50360, 50361], 'begin_suppress_tokens': [220, 50256]}


model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

events.out.tfevents.1734626346.55df2263cf82.40.2:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

events.out.tfevents.1734621808.55df2263cf82.40.1:   0%|          | 0.00/14.6k [00:00<?, ?B/s]

events.out.tfevents.1734620390.55df2263cf82.40.0:   0%|          | 0.00/6.67k [00:00<?, ?B/s]

...

events.out.tfevents.1734631888.55df2263cf82.40.7:   0%|          | 0.00/8.99k [00:00<?, ?B/s]

events.out.tfevents.1734631708.55df2263cf82.40.6:   0%|          | 0.00/459 [00:00<?, ?B/s]

events.out.tfevents.1734633581.55df2263cf82.40.8:   0%|          | 0.00/9.92k [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.37k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/khizarAI/finetune-whisper-base.en/commit/693732480d9802811c87b0e25cfa990a764f8868', commit_message='End of training', commit_description='', oid='693732480d9802811c87b0e25cfa990a764f8868', pr_url=None, pr_revision=None, pr_num=None)

In [77]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("automatic-speech-recognition", model="khizarAI/finetune-whisper-base.en")

config.json:   0%|          | 0.00/2.23k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/999k [00:00<?, ?B/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.17k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [78]:
output = pipe("/kaggle/input/pakistani-english-dataset/WAV/G0001/G0001_0_S0108.wav")

/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:496: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


In [79]:
output

{'text': 'we have to proceed and have to continue'}